# Evaluator — Quickstart Notebook

End-to-end walkthrough of the `evaluator` library for audio retrieval evaluation.

**Pipeline:** audio query → ASR transcription → text embedding → corpus retrieval → metrics

**Sections:**
1. Install
2. Prepare sample data
3. Run with a preset (one-liner)
4. Build a custom config
5. Inspect results
6. Compare two models
7. Grid search over retrieval params
8. Save / load results

## 1. Install

In [ ]:
# Install the evaluator package (base dependencies only)
%pip install -q -e '..'  # from repo root
# For remote install:
# %pip install -q evaluator

# Optional: datasets library to download sample data from HuggingFace
%pip install -q datasets soundfile

## 2. Prepare sample data

Evaluator expects two JSON files:
- **`questions.json`** — queries with ground-truth doc IDs
- **`corpus.json`** — document collection to retrieve from

We use PubMedQA (medical Q&A) as a small realistic example.

In [ ]:
from pathlib import Path
import json

DATA_DIR = Path("data/pubmed_qa_small")
QUESTIONS_PATH = DATA_DIR / "questions.json"
CORPUS_PATH    = DATA_DIR / "corpus.json"

if QUESTIONS_PATH.exists() and CORPUS_PATH.exists():
    print("Data already present — skipping download.")
else:
    from datasets import load_dataset

    N_SAMPLES = 20
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    hf_ds = load_dataset("pubmed_qa", "pqa_labeled", split=f"train[:{N_SAMPLES}]")

    questions, corpus = [], []
    for row in hf_ds:
        doc_id = str(row["pubid"])
        context = "\n\n".join(c.strip() for c in row["context"]["contexts"] if c.strip())
        if row.get("long_answer"):
            context = f"{context}\n\n{row['long_answer'].strip()}" if context else row["long_answer"].strip()

        corpus.append({
            "doc_id":   doc_id,
            "title":    row["question"],
            "text":     context,
            "language": "en",
            "metadata": {"pubid": row["pubid"], "decision": row.get("final_decision")},
        })
        questions.append({
            "question_id":         f"q_{doc_id}",
            "question_text":       row["question"],
            "groundtruth_doc_ids": [doc_id],
            "relevance_grades":    {doc_id: 1},
            "language":            "en",
            "metadata":            {"pubid": row["pubid"]},
        })

    QUESTIONS_PATH.write_text(json.dumps(questions, indent=2))
    CORPUS_PATH.write_text(json.dumps(corpus, indent=2))
    print(f"Saved {len(questions)} questions and {len(corpus)} corpus docs.")

questions = json.loads(QUESTIONS_PATH.read_text())
corpus    = json.loads(CORPUS_PATH.read_text())
print(f"Questions: {len(questions)}  |  Corpus docs: {len(corpus)}")
print("Sample question:", questions[0]["question_text"])

## 3. Run with a preset — one-liner

Presets are pre-configured model combinations. Available: `whisper_labse`, `wav2vec_jina`, `audio_only`, `fast_dev`.

In [ ]:
from evaluator import list_presets, evaluate_from_preset

print("Available presets:", list_presets())

In [ ]:
# fast_dev: whisper-tiny + LaBSE, CPU, small batch — ideal for quick testing
results = evaluate_from_preset(
    "fast_dev",
    data_path=str(QUESTIONS_PATH),
    corpus_path=str(CORPUS_PATH),
    # override any preset field with underscore notation:
    data_trace_limit=5,     # only process first 5 queries
    data_batch_size=2,
    model_asr_device="cpu",
    model_text_emb_device="cpu",
    cache_enabled=True,
    cache_cache_dir=str(DATA_DIR / "cache"),
)

print(results)

## 4. Build a custom config

`EvaluationConfig` is the central config object. Build from a dict, YAML, or preset.

In [ ]:
from evaluator import EvaluationConfig, run_evaluation

cfg = EvaluationConfig.from_dict({
    "experiment_name": "my_whisper_labse_run",
    "output_dir":      str(DATA_DIR / "results"),

    "model": {
        "pipeline_mode":       "asr_text_retrieval",  # audio → ASR → text embed → retrieve
        "asr_model_type":      "whisper",
        "asr_model_name":      "openai/whisper-tiny",
        "asr_device":          "cpu",
        "text_emb_model_type": "labse",
        "text_emb_model_name": "sentence-transformers/LaBSE",
        "text_emb_device":     "cpu",
    },

    "data": {
        "questions_path":   str(QUESTIONS_PATH),
        "corpus_path":      str(CORPUS_PATH),
        "batch_size":       2,
        "trace_limit":      5,
        "strict_validation": False,
    },

    "vector_db": {
        "type":                  "inmemory",
        "k":                     5,
        "retrieval_mode":        "dense",
    },

    "audio_synthesis": {
        "enabled":    True,
        "provider":   "mms",
        "language":   "en",
        "output_dir": str(DATA_DIR / "audio"),
    },

    "cache": {
        "enabled":   True,
        "cache_dir": str(DATA_DIR / "cache"),
    },

    "service_runtime": {
        "startup_mode":   "lazy",
        "offload_policy": "on_finish",
    },

    "tracking": {"enabled": False},
})

# Inspect config sections
print("Pipeline mode:", cfg.model.pipeline_mode)
print("ASR model:    ", cfg.model.asr_model_name)
print("Embedding:    ", cfg.model.text_emb_model_name)
print("Retrieval k:  ", cfg.vector_db.k)

In [ ]:
# Save config to YAML for reproducibility
cfg.to_yaml(str(DATA_DIR / "my_config.yaml"))
print("Config saved.")

# Round-trip: reload from YAML
cfg_reloaded = EvaluationConfig.from_yaml(str(DATA_DIR / "my_config.yaml"))
print("Reloaded experiment name:", cfg_reloaded.experiment_name)

In [ ]:
results_custom = run_evaluation(cfg)
print(results_custom)

## 5. Inspect results

In [ ]:
# All metrics
print("All metrics:")
for name, value in sorted(results_custom.metrics.items()):
    if isinstance(value, float):
        print(f"  {name:<20} {value:.4f}")
    else:
        print(f"  {name:<20} {value}")

In [ ]:
# Key metrics by name
mrr     = results_custom.get_metric("MRR", default=0.0)
recall5 = results_custom.get_metric("Recall@5", default=0.0)
wer     = results_custom.get_metric("WER", default=0.0)

print(f"MRR:      {mrr:.4f}")
print(f"Recall@5: {recall5:.4f}")
print(f"WER:      {wer:.2%}")

# One-line summary
print("\nSummary:", results_custom.summary())

In [ ]:
# Metadata: timing, run info
meta = results_custom.metadata
print(f"Duration:    {meta.get('duration_seconds', 0):.1f}s")
print(f"Num samples: {meta.get('num_samples', '?')}")
print(f"Created at:  {meta.get('created_at', '?')}")

## 6. Compare two models

Run a small comparison between `dense` and `hybrid` retrieval modes.

In [ ]:
from evaluator import run_evaluation_matrix
from dataclasses import replace

# Build variant configs from base
cfg_dense = replace(cfg, experiment_name="dense_retrieval")

cfg_hybrid = EvaluationConfig.from_dict({
    **cfg.to_dict(),
    "experiment_name": "hybrid_retrieval",
    "vector_db": {
        **cfg.to_dict()["vector_db"],
        "retrieval_mode":         "hybrid",
        "hybrid_dense_weight":    0.6,
        "hybrid_fusion_method":   "rrf",
    },
})

all_results = run_evaluation_matrix([cfg_dense, cfg_hybrid])

print(f"{'Experiment':<25} {'MRR':>8} {'Recall@5':>10} {'WER':>8}")
print("-" * 55)
for r in all_results:
    name = r.config.experiment_name
    mrr  = r.get_metric("MRR", 0.0)
    r5   = r.get_metric("Recall@5", 0.0)
    wer  = r.get_metric("WER", 0.0)
    print(f"{name:<25} {mrr:>8.4f} {r5:>10.4f} {wer:>8.2%}")

## 7. Grid search over retrieval params

`GridSearch` systematically sweeps config parameters to find the best combination.

In [ ]:
from evaluator.analysis.grid_search import GridSearch, run_grid_search, analyze_grid_results

# Define the grid: vary k and dense weight
grid = GridSearch(cfg)
grid.add_param("vector_db.k", [3, 5, 10])

print(f"Grid size: {grid.get_size()} configurations")
print(grid.summary())

In [ ]:
def eval_fn(config):
    """Wrapper: run evaluation and return flat metrics dict."""
    r = run_evaluation(config)
    return r.metrics

grid_results = run_grid_search(grid, eval_fn)

analysis = analyze_grid_results(grid_results, metric_name="MRR", top_k=3)

print("Best config:")
print(f"  params:  {analysis['best_config']['params']}")
print(f"  MRR:     {analysis['best_config']['MRR']:.4f}")
print()
print("Top 3 configs:")
for entry in analysis["top_configs"]:
    print(f"  #{entry['rank']}  params={entry['params']}  MRR={entry['MRR']:.4f}")
print()
stats = analysis["summary"]["metric_stats"]
print(f"MRR range: {stats['min']:.4f} – {stats['max']:.4f}  (mean {stats['mean']:.4f}, std {stats['std']:.4f})")

## 8. Save and load results

In [ ]:
from evaluator import EvaluationResults

RESULTS_PATH = DATA_DIR / "results" / "my_run.json"

# Save
results_custom.save(str(RESULTS_PATH))
print(f"Saved to {RESULTS_PATH}")

# Load
loaded = EvaluationResults.load(str(RESULTS_PATH))
print("Loaded:", loaded.summary())

# Confirm round-trip integrity
assert loaded.metrics == results_custom.metrics, "Metrics mismatch after round-trip!"
print("Round-trip OK.")